### Day 09 · OOP 继承 (L2)

今天进入面向对象三大特性的第一个:**继承**。
你会学会:复用已有类的代码、重写父类行为、理解查找顺序。

> 叙事锚点:电商订单系统 v2 —— 实体书/电子书继承 Product 基类

#### 为什么需要继承(is-a 关系)

> **痛点**  
写几个类发现彼此有大量重复代码,比如实体书和电子书几乎完全一样,复制粘贴越多,改 BUG 越痛。

> **类比**  
动物 → 猫 / 狗:猫 'is-a' 动物,继承了动物的共同特征,再扩展自己的'叫'。

> **解释**  
继承让子类**自动拥有**父类的属性和方法,在此基础上添加或修改,实现代码复用。

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name
    def breathe(self):
        print(f"{self.name} 在呼吸")

class Dog(Animal):
    def bark(self):
        print(f"{self.name} 汪汪叫")

dog = Dog("旺财")
dog.breathe()   # 继承自 Animal
dog.bark()      # 子类自己的方法

> **逐行解剖**  
① `class Animal` 定义父类  
② `class Dog(Animal)` Dog 继承 Animal 的所有成员  
③ `dog = Dog('旺财')` → Dog 没有 __init__,沿'子类→父类'找到 Animal.__init__  
④ `dog.breathe()` → Dog 没有 breathe,找到 Animal.breathe

**练习** — 定义 Vehicle 继承体系

定义父类 `Vehicle`,属性 `brand`,方法 `start()` 输出 'XXX 启动'。
定义子类 `Car`,新增方法 `honk()`。

> 问自己:子类括号里应该写什么?

In [ ]:
# ============ 学员代码区 ============
class Vehicle:
    pass

# car = Car('比亚迪')
# car.start()
# car.honk()
pass

# ============ 参考答案 ============
class Vehicle:
    def __init__(self, brand):
        self.brand = brand
    def start(self):
        print(f"{self.brand} 启动")

class Car(Vehicle):
    def honk(self):
        print(f"{self.brand} 按喇叭")

car = Car('比亚迪')
car.start()
car.honk()

#### super().__init__() 扩展父类属性

> **痛点**  
子类想扩展父类属性,又不想重复写父类的绑定逻辑。

> **类比**  
儿子继承了父亲的房子(父类属性),再加盖一层(扩展属性),但必须**先**把父亲的房子盖好。

> **解释**  
`super()` 返回父类对象,用来**安全地调用父类方法**。

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)
        self.breed = breed

dog = Dog("旺财", "柯基")
print(dog.name, dog.breed)
dog2 = Dog("来福", "柴犬")
print(dog2.name)  # 来福(独立)

> **逐行解剖**  
① `super().__init__(name)` → 调用 Animal.__init__ → self.name 绑好  
② `self.breed = breed` 绑子类自己的属性

**练习** — 子类扩展属性

定义 `Product` 基类,属性 `name` 和 `price`。
定义子类 `Book(Product)`,新增属性 `author`。
用 `super().__init__()` 继承基类属性。

In [ ]:
# ============ 学员代码区 ============
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

class Book(Product):
    pass

# b = Book("Python 入门", 59.8, "张三")
# print(b.name, b.price, b.author)
pass

# ============ 参考答案 ============
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

class Book(Product):
    def __init__(self, name, price, author):
        super().__init__(name, price)
        self.author = author

b = Book("Python 入门", 59.8, "张三")
print(b.name, b.price, b.author)

#### 方法重写(override)

> **痛点**  
父类的方法不能满足子类需要,比如基类 `shipping_cost()` 返回 0,但实体书要运费。

> **解释**  
子类重新定义同名方法,调用时**优先执行子类版本**。

In [ ]:
class Product:
    def shipping_cost(self):
        return 0

class PhysicalProduct(Product):
    def __init__(self, name, price, weight):
        super().__init__()
        self.name, self.price, self.weight = name, price, weight
    def shipping_cost(self):
        return self.weight * 5

class DigitalProduct(Product):
    pass  # 不重写,继承基类的 0

p = PhysicalProduct("纸质书", 50, 2)
d = DigitalProduct()
print(p.shipping_cost())  # 10
print(d.shipping_cost())  # 0

**练习** — 电商产品运费体系

定义 `Product` 基类,方法 `shipping_cost()` 返回 0。
子类 `PhysicalProduct` 新增 `weight`,重写 `shipping_cost()` = weight * 8。
子类 `DigitalProduct` 不重写。各创建一个验证。

In [ ]:
# ============ 学员代码区 ============
class Product:
    def shipping_cost(self):
        return 0

class PhysicalProduct(Product):
    pass

# pp = PhysicalProduct(10)
# print(pp.shipping_cost())
pass

# ============ 参考答案 ============
class Product:
    def shipping_cost(self):
        return 0

class PhysicalProduct(Product):
    def __init__(self, weight): self.weight = weight
    def shipping_cost(self):
        return self.weight * 8

class DigitalProduct(Product):
    pass

pp = PhysicalProduct(10)
dp = DigitalProduct()
print(pp.shipping_cost())  # 80
print(dp.shipping_cost())  # 0

#### MRO + isinstance 反模式

> **解释**  
- 查找顺序:子类 → 父类 → 父类的父类 → object
- `isinstance` 可以做类型判断,但**用它做'类型分发'是反模式**

In [ ]:
class Animal: pass
class Mammal(Animal): pass
class Dog(Mammal): pass

print(Dog.__mro__)
print(isinstance(Dog(), Animal))  # True(继承链)

**练习** — MRO + isinstance

定义 `Animal→Mammal→Dog` 三级继承。
查看 `Dog.__mro__`。
验证 `isinstance(dog, Animal)` 沿继承链追溯。

In [ ]:
# ============ 学员代码区 ============
class Animal: pass

# class Mammal: pass
# class Dog: pass
# print(Dog.__mro__)
pass

# ============ 参考答案 ============
class Animal: pass
class Mammal(Animal): pass
class Dog(Mammal): pass

print(Dog.__mro__)
dog = Dog()
print(isinstance(dog, Animal))  # True
print(isinstance(dog, Mammal))  # True

#### 综合练习:员工薪资系统

1. 基类 `Employee(name, base)`,方法 `pay()` 返回 base
2. 子类 `Sales(name, base, commission)`,重写 pay()
3. 子类 `Manager(name, base, bonus)`,重写 pay()
4. 用 `super().__init__()` 调用父类构造

In [ ]:
# ============ 学员代码区 ============
class Employee:
    def __init__(self, name, base):
        self.name = name
        self.base = base
    def pay(self):
        return self.base

# class Sales(Employee): ...
# class Manager(Employee): ...
# emps = [Employee("张三", 5000), Sales("李四", 5000, 2000)]
# for e in emps: print(e.pay())
pass

# ============ 参考答案 ============
class Employee:
    def __init__(self, name, base):
        self.name = name
        self.base = base
    def pay(self):
        return self.base

class Sales(Employee):
    def __init__(self, name, base, commission):
        super().__init__(name, base)
        self.commission = commission
    def pay(self):
        return super().pay() + self.commission

class Manager(Employee):
    def __init__(self, name, base, bonus):
        super().__init__(name, base)
        self.bonus = bonus
    def pay(self):
        return super().pay() + self.bonus

for e in [Employee("张三", 5000), Sales("李四", 5000, 2000)]:
    print(e.pay())

**今日小结**

| 概念 | 解决的痛点 |
| --- | --- |
| 继承 | 复用已有类代码,避免重复 |
| super() | 安全调用父类构造函数 |
| 方法重写 | 子类用自定义行为替换父类 |
| MRO | 多级继承时查找顺序清晰 |

**更多练习**
- 当堂练:course/lesson09/in_class/practice01-06.py
- 课后作业:course/lesson09/homework/task01-03.py